# 06 · Evaluation & Regression Tests
Automated evaluation over the synthetic academic benchmark, testing answer correctness, grounding, citation accuracy, abstention rates, and running the full pytest suite.

In [ ]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
from dotenv import load_dotenv
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# Automatically load environment variables from rag_service/.env and repo-root .env
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / ".env", override=False)

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env (default when keys are present)
has_keys = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY"))
OFFLINE = (os.getenv("NB_OFFLINE", "0" if has_keys else "1") == "1")
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings, reset_settings_cache
reset_settings_cache()
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

## 1. Benchmark Evaluation on Synthetic QA Set

In [ ]:
import synthetic, pandas as pd
from app.agent import ask, NO_ANSWER
from app.store import get_store
assert get_store().count() > 0, "Run notebook 02 first"
rows = []
for subj, q, kw, src in synthetic.QA:
    o = ask(q, subject_codes=[subj])
    rows.append(dict(subject=subj, question=q, correct=kw in o["answer"].lower(), grounded=o["grounded"],
                     cited=bool(o["sources"]), conf=o["confidence"], ms=o["latency_ms"]))
ans = pd.DataFrame(rows); ans

In [ ]:
print("Answer Correctness :", round(ans.correct.mean(), 3))
print("Groundedness       :", round(ans.grounded.mean(), 3))
print("Citation Presence  :", round(ans.cited.mean(), 3))
print("Mean Confidence    :", round(ans.conf.mean(), 3), "| Mean Latency ms:", int(ans.ms.mean()))

## 2. Unanswerable Query Abstention Rate

In [ ]:
un = [ask(q, subject_codes=[s]) for s, q in synthetic.UNANSWERABLE]
print("Abstained correctly:", sum(o["answer"] == NO_ANSWER for o in un), "/", len(un))

## 3. Cross-Subject Leakage Probes

In [ ]:
leaks = [ask(q, subject_codes=[s]) for s, q, _ in synthetic.LEAKAGE]
print("Zero leakage verified:", all(o["answer"] == NO_ANSWER and not o["sources"] for o in leaks))

## 4. Run Pytest Regression Suite

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-600:])